In [1]:
# Install libraries required for the RAG project

!pip install -q \
pypdf \
langchain \
langchain-community \
langchain-text-splitters \
chromadb \
sentence-transformers \
transformers \
accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

In [2]:
# Import libraries required for document loading, text processing, embeddings, vector storage, and LLM generation

import os
import torch

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

from transformers import pipeline

In [3]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Verify access to the ADA guideline documents and identify the PDF files

data_path = "/content/drive/MyDrive/ADA_Diabetes_Guidelines"

pdf_files = sorted(
    [file for file in os.listdir(data_path) if file.endswith(".pdf")]
)

print("PDF files found:")
for pdf in pdf_files:
    print(pdf)

PDF files found:
dc26s002.pdf
dc26s004.pdf
dc26s005.pdf
dc26s006.pdf
dc26s009.pdf
dc26s010.pdf


In [5]:
# Load the ADA guideline PDF documents and check the number of pages

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)
    print(f"{pdf}: {len(reader.pages)} pages")

dc26s002.pdf: 23 pages
dc26s004.pdf: 28 pages
dc26s005.pdf: 43 pages
dc26s006.pdf: 18 pages
dc26s009.pdf: 33 pages
dc26s010.pdf: 30 pages


In [6]:
# Extract text from the ADA guideline PDF documents

documents = []

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append(
        {
            "filename": pdf,
            "text": text
        }
    )

print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [7]:
# Display basic information about the extracted documents

for document in documents:
    print("=" * 60)
    print(document["filename"])
    print(f"Characters: {len(document['text'])}")

dc26s002.pdf
Characters: 165958
dc26s004.pdf
Characters: 195695
dc26s005.pdf
Characters: 338666
dc26s006.pdf
Characters: 128520
dc26s009.pdf
Characters: 223271
dc26s010.pdf
Characters: 222366


In [8]:
# Preview the extracted text from the first ADA guideline document

print(documents[0]["text"][:2000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES
2. Diagnosis and Classification of
Diabetes: Standards of Care in
Diabetes—2026
Diabetes Care 2026;49(Suppl. 1):S27–S49 | https://doi.org/10.2337/dc26-S002
American Diabetes Association 
Professional Practice 
Committee for 
Diabetes*
*A complete list of members of the American 
Diabetes Association 
Professional Practice
Committee for Diabetes can be found at https:// 
doi.org/10.2337/dc26-SINT.
Duality of interes t information for each contributor 
is available at https://doi.org/10.2337/dc26-SDIS.
Suggested citation: American Diabetes Association 
Professional 
Practice Committee for Diabetes. 
2. Diagnosis and classification of diabetes: 
Standards of Care in Diabetes—2026. Diabetes Care 
2026;49(Suppl. 1):S27–S49
© 2025 by the American Diabetes Association. 
Readers may 
use this work for educational, 
noncommercial purposes if properly cited and 
unaltered. The version of record may be linked 
at https://diabetesjournals.org/care, but A

In [9]:
# Clean extracted ADA guideline text before chunking

import re

def clean_guideline_text(text):
    # Join broken line breaks into spaces
    text = text.replace("\n", " ")

    # Remove downloaded-from text
    text = re.sub(r"Downloaded from .*?(?=S\d+|$)", " ", text)

    # Remove journal/header/footer phrases
    text = re.sub(r"Diabetes Care Volume 49, Supplement 1, January 2026", " ", text)
    text = re.sub(r"diabetesjournals\.org/care", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_documents = []

for document in documents:
    cleaned_text = clean_guideline_text(document["text"])
    cleaned_documents.append(
        {
            "filename": document["filename"],
            "text": cleaned_text
        }
    )

print(f"Cleaned {len(cleaned_documents)} documents.")

Cleaned 6 documents.


In [10]:
# Preview the cleaned text from the first document

print(cleaned_documents[0]["text"][:3000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES 2. Diagnosis and Classification of Diabetes: Standards of Care in Diabetes—2026 Diabetes Care 2026;49(Suppl. 1):S27–S49 | American Diabetes Association Professional Practice Committee for Diabetes* *A complete list of members of the American Diabetes Association Professional Practice Committee for Diabetes can be found at https:// doi.org/10.2337/dc26-SINT. Duality of interes t information for each contributor is available at Suggested citation: American Diabetes Association Professional Practice Committee for Diabetes. 2. Diagnosis and classification of diabetes: Standards of Care in Diabetes—2026. Diabetes Care 2026;49(Suppl. 1):S27–S49 © 2025 by the American Diabetes Association. Readers may use this work for educational, noncommercial purposes if properly cited and unaltered. The version of record may be linked at https:// , but ADA permission is required to post this work on any third-party site or platform. This publication and its cont

In [11]:
# Split cleaned ADA guideline text into smaller chunks for retrieval

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for document in cleaned_documents:
    document_chunks = text_splitter.create_documents(
        [document["text"]],
        metadatas=[{"source": document["filename"]}]
    )
    chunks.extend(document_chunks)

print(f"Total cleaned chunks created: {len(chunks)}")

Total cleaned chunks created: 1531


In [12]:
# Preview a representative text chunk

print("Source:", chunks[10].metadata["source"])
print("\nChunk preview:\n")
print(chunks[10].page_content)

Source: dc26s002.pdf

Chunk preview:

. In this con- text, testing A1C helps determine the chronicity of hyperglycemia. However, in an individual without symptoms, FPG or 2-h PG can be used for screening and di - agnosis of diabetes. In nonpregnant indi- viduals, FPG (or A1C) is typically preferred for routine screening due to the ease of administration ( Table 2.3); however, the 2-h PG (OGTT) testing protocol is signifi- cantly more sensitive than the other two tests and is preferentially recommended for screening for some conditions (e.g., cystic fibrosis –related diabetes or post - transplantation diabetes mellitus). In the absence of classic symptoms of hypergly- cemia, repeat testing is required to con- firm the diagnosis regardless of the test used (see CONFIRMING THE DIAGNOSIS, below). Major advantages of glucose monitor- ing are its low cost and availability. Dis - advantages include the high diurnal variation in glucose and the 8-h fasting requirements ( Table 2.3)


In [13]:
# Load the embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [14]:
# Generate embeddings for all text chunks

chunk_texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print(f"Generated embeddings for {len(embeddings)} chunks.")
print(f"Embedding dimension: {len(embeddings[0])}")

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Generated embeddings for 1531 chunks.
Embedding dimension: 384


In [15]:
# Create a ChromaDB client and collection

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="ada_diabetes_guidelines_cleaned"
)

print("ChromaDB collection created successfully.")

ChromaDB collection created successfully.


In [16]:
# Store text chunks, embeddings, and metadata in ChromaDB

ids = [f"chunk_{i}" for i in range(len(chunks))]
documents_text = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 1531 chunks in ChromaDB.


In [17]:
# Test basic retrieval using a sample Type 2 Diabetes management question

query = "What is the recommended HbA1c target for most adults with Type 2 Diabetes?"

query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print("Question:", query)
print("\nTop retrieved results:\n")

for i, document in enumerate(results["documents"][0]):
    print("=" * 80)
    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i]["source"])
    print(document[:1000])

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved results:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of diabetes me

In [18]:
# Create a reusable function for retrieving relevant guideline chunks

def retrieve_guideline_chunks(query, n_results=3):
    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    print("Question:", query)
    print("\nTop retrieved guideline chunks:\n")

    for i, document in enumerate(results["documents"][0]):
        print("=" * 80)
        print(f"Result {i + 1}")
        print("Source:", results["metadatas"][0][i]["source"])
        print(document[:1000])
        print()

In [19]:
# Test retrieval using multiple Type 2 Diabetes management questions

sample_questions = [
    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
    "What medications are recommended for Type 2 Diabetes treatment?",
    "How should cardiovascular risk be managed in people with diabetes?",
    "What lifestyle changes are recommended for Type 2 Diabetes?",
    "How is diabetes diagnosed?"
]

for question in sample_questions:
    retrieve_guideline_chunks(question, n_results=3)
    print("\n\n")

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved guideline chunks:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of di

In [20]:
# Load an open-source Hugging Face language model

import torch
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

print("Open-source language model loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Open-source language model loaded successfully.


## Baseline Qwen RAG Answer Generation

The retrieval pipeline was integrated with the open-source instruction-tuned language model, Qwen2.5-3B-Instruct. This allowed the system to move beyond simply retrieving document chunks and generate natural-language answers using the relevant ADA Standards of Care in Diabetes—2026 guideline excerpts.

### Observation

The baseline Qwen output showed that the end-to-end RAG pipeline was working and could generate answers based on the retrieved guideline excerpts. However, after answering the question, the model generated an unnecessary summary that repeated key points already presented. The response also included supporting research details rather than focusing mainly on the ADA recommendations and used content from multiple guideline chapters. These observations guided the improvements made in the following experiments.

In [21]:
# Verify that the QWEN2.5-3B language model is loaded and generating responses

prompt = "Briefly explain what a Retrieval-Augmented Generation (RAG) system does."

response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(response[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Briefly explain what a Retrieval-Augmented Generation (RAG) system does. A Retrieval-Augmented Generation (RAG) system is an AI model that combines the strengths of retrieval and generation to improve its performance on tasks involving large amounts of unstructured data. It works by first retrieving relevant documents from a knowledge base or corpus, then using those retrieved documents as context to generate a response to the user's query. This approach allows the model to access a vast amount of information beyond just its pre-trained language model, making it better equipped to understand complex queries and provide accurate answers


In [22]:
# Generate a RAG answer using retrieved ADA guideline chunks and QWEN2.5-3B

def generate_rag_answer_baseline(question, n_results=3):

    # Generate an embedding for the user's question
    question_embedding = embedding_model.encode(question).tolist()

    # Retrieve the most relevant document chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    # Combine the retrieved chunks into a single context
    context = "\n\n".join(
        [
            f"Source: {retrieved_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(retrieved_chunks)
        ]
    )

    # Build the prompt
    prompt = f"""
You are a clinical decision support assistant.

Answer the user's question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts provided below.

Instructions:
- Do NOT use outside knowledge.
- Do NOT guess or add information that is not present in the retrieved excerpts.
- If the retrieved excerpts only partially answer the question, clearly state that.
- Answer in 3–5 concise bullet points.
- Use clear, professional language suitable for a healthcare decision-support system.
- Mention the source document(s) used at the end of your answer.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    # Generate an answer using QWEN2.5-3B
    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    # Display the retrieved source documents
    sources_used = sorted(set([source["source"] for source in retrieved_sources]))

    print("Question:", question)
    print("\nSources retrieved:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [23]:
# Test the baseline RAG Pipeline (QWEN2.5-3B)


_ = generate_rag_answer_baseline(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    n_results=3
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources retrieved:
- dc26s009.pdf

Generated answer:

- The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications.
- The choice should aim to achieve and maintain individualized glycemic goals while considering cardiovascular, kidney, weight, and other comorbidities.
- Hypoglycemia risk, cost, access, adverse reactions, and individual preferences are also important factors to consider.
- Combination therapy may be considered for initial treatment to expedite reaching individualized glycemic goals. Source: dc26s009.pdf

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The choice should aim to achieve and maintain individualized glycemic goals while considering cardiovascular, kidney, weight, and other comorbidities. Hypoglycemia risk

## Prompt Refinement

To improve the baseline response, I refined the prompt to reduce repetition, keep the answers concise, focus on ADA recommendations, and prevent the model from using information outside the retrieved guideline excerpts.

### Observation

The refined prompt produced a clearer and more concise answer, and the unnecessary summary seen in the baseline response was no longer present. However, the model still emphasized specific medication examples from the retrieved excerpts instead of focusing mainly on the overall ADA recommendation. This showed that refining the prompt alone was not enough and that the retrieval process also needed further improvement.

In [24]:
# Generate a RAG answer using a refined QWEN2.5-3B prompt

def generate_rag_answer_refined(question, n_results=3):

    # Generate an embedding for the user's question
    question_embedding = embedding_model.encode(question).tolist()

    # Retrieve the most relevant document chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    # Combine retrieved chunks with source labels
    context = "\n\n".join(
        [
            f"Source: {retrieved_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(retrieved_chunks)
        ]
    )

    # Refined prompt
    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do NOT use outside knowledge.
2. Do NOT repeat yourself.
3. Do NOT summarize twice.
4. Focus on ADA recommendations, not supporting research studies or reference lists.
5. If the retrieved excerpts do not fully answer the question, state that clearly.
6. Keep the answer under 150 words.
7. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using QWEN2.5-3B
    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    # Display retrieved sources
    sources_used = sorted(set([source["source"] for source in retrieved_sources]))

    print("Question:", question)
    print("\nSources retrieved:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [25]:
# Test the refined QWEN2.5-3B RAG prompt

_ = generate_rag_answer_refined(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    n_results=3
)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources retrieved:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The goal is to select treatments that effectively help achieve and maintain individualized glycemic goals while considering various factors such as cardiovascular risks, kidney function, weight impact, hypoglycemia risk, cost, and patient preferences. Combination therapy may be considered initially to expedite reaching these goals. Source: dc26s009.pdf
source: dc26s009.pdf


## Filtering Reference-Heavy Chunks

The next improvement was to filter out retrieved chunks that appeared to contain mostly references, citation lists, or journal metadata before sending the context to the language model.

### Observation

Filtering reference-heavy chunks helped reduce some of the reference-heavy context, but it did not completely solve the issue. The generated answer still used content from multiple guideline chapters and included specific medication examples rather than focusing only on the main ADA recommendation. The output also included a small formatting artifact at the end. This showed that chunk filtering helped, but source selection and output formatting still needed further refinement.

In [26]:
# Filter retrieved chunks to reduce reference-heavy context

def is_reference_heavy(chunk):
    reference_terms = [
        "et al.",
        "Diabetes Care",
        "Lancet",
        "BMJ",
        "Clin Chem",
        "Ann Intern Med",
        "Accessed",
        "Available from",
        "doi",
        "References"
    ]

    count = sum(term.lower() in chunk.lower() for term in reference_terms)

    return count >= 3

In [27]:
# Generate a RAG answer using QWEN2.5-3B with filtered retrieved chunks

def generate_rag_answer_filtered(question, n_results=5):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    filtered_chunks = []
    filtered_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        if not is_reference_heavy(chunk):
            filtered_chunks.append(chunk)
            filtered_sources.append(source)

    if len(filtered_chunks) == 0:
        filtered_chunks = retrieved_chunks
        filtered_sources = retrieved_sources

    context = "\n\n".join(
        [
            f"Source: {filtered_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(filtered_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do NOT use outside knowledge.
2. Do NOT repeat yourself.
3. Focus on ADA recommendations, not supporting research studies or reference lists.
4. Do NOT list medication names unless they are clearly presented as recommendations in the retrieved excerpts.
5. If the retrieved excerpts only partially answer the question, state that clearly.
6. Keep the answer under 150 words.
7. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    sources_used = sorted(set([source["source"] for source in filtered_sources]))

    print("Question:", question)
    print("\nSources used after filtering:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [28]:
# Test filtered RAG answer

_ = generate_rag_answer_filtered(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    n_results=5
)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used after filtering:
- dc26s006.pdf
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain intended treatment goals while considering cardiovascular, kidney, weight, and other relevant comorbidities; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy might be considered initially to help reach individualized glycemic goals. Source: dc26s009.pdf. To facilitate positive health behaviors and improve outcomes, lifestyle modifications should also be emphasized alongside any pharmacologic therapy. Source: dc26s009.pdf.


## Source-Aware Retrieval

For the medication question, I tested whether focusing on the most relevant ADA chapter would improve the generated response. Since pharmacologic treatment is mainly covered in Chapter 9, I filtered the retrieved context to prioritize `dc26s009.pdf`.

### Observation

Restricting the context to the pharmacologic treatment chapter produced a more focused answer. However, the original medication question still encouraged the model to look for a list of drugs, which led to some study-specific details. Rephrasing the question to better match the guideline wording improved the output. Asking how the ADA recommends selecting glucose-lowering medications produced a clearer answer than asking for a simple list of recommended medications.

In [29]:
# Generate a RAG answer filtered to the main pharmacologic treatment chapter

def generate_rag_answer_source_filtered(question, target_source="dc26s009.pdf", n_results=8):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    selected_chunks = []
    selected_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        if source["source"] == target_source and not is_reference_heavy(chunk):
            selected_chunks.append(chunk)
            selected_sources.append(source)

    if len(selected_chunks) == 0:
        selected_chunks = retrieved_chunks[:3]
        selected_sources = retrieved_sources[:3]

    context = "\n\n".join(
        [
            f"Source: {selected_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(selected_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
- Do not use outside knowledge.
- Do not repeat yourself.
- Focus on ADA recommendations.
- Do not list medication names unless the excerpts clearly present them as recommendations.
- If the excerpts only partially answer the question, say that clearly.
- Keep the answer under 150 words.
- End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=160,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    print("Question:", question)
    print("\nSources used:")
    for source in sorted(set([s["source"] for s in selected_sources])):
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [30]:
# Test source-filtered RAG answer for original medication question

_ = generate_rag_answer_source_filtered(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    target_source="dc26s009.pdf",
    n_results=8
)

[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain individualized glycemic goals while considering factors such as cardiovascular, kidney, weight, and other comorbidities' effects; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy might be considered initially to expedite reaching these goals. Source: dc26s009.pdf
```


## Final Improved RAG Pipeline (QWEN2.5-3B)

Based on the experiments above, the final Week 4 pipeline combines semantic retrieval, reference filtering, prompt refinement, and simple source-aware routing. Together, these improvements help the language model generate more focused answers from the retrieved ADA guideline excerpts.

In [31]:
# Final improved RAG pipeline (QWEN2.5-3B)

def select_target_source(question):
    question_lower = question.lower()

    if any(term in question_lower for term in ["medication", "drug", "glucose-lowering", "pharmacologic"]):
        return "dc26s009.pdf"
    elif any(term in question_lower for term in ["diagnosis", "diagnosed", "diagnose"]):
        return "dc26s002.pdf"
    elif any(term in question_lower for term in ["hba1c", "a1c", "glycemic target", "glycemic goal"]):
        return "dc26s006.pdf"
    elif any(term in question_lower for term in ["cardiovascular", "heart", "cvd", "risk management"]):
        return "dc26s010.pdf"
    elif any(term in question_lower for term in ["lifestyle", "physical activity", "exercise", "sleep", "self-management"]):
        return "dc26s005.pdf"
    else:
        return None


def generate_rag_answer(question, n_results=8):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    target_source = select_target_source(question)

    selected_chunks = []
    selected_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        source_name = source["source"]

        if is_reference_heavy(chunk):
            continue

        if target_source is None or source_name == target_source:
            selected_chunks.append(chunk)
            selected_sources.append(source)

    if len(selected_chunks) == 0:
        selected_chunks = retrieved_chunks[:3]
        selected_sources = retrieved_sources[:3]

    context = "\n\n".join(
        [
            f"Source: {selected_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(selected_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do not use outside knowledge.
2. Do not repeat yourself.
3. Focus on ADA recommendations, not supporting studies or reference lists.
4. If the retrieved excerpts only partially answer the question, state that clearly.
5. Keep the answer concise and professional.
6. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    sources_used = sorted(set([source["source"] for source in selected_sources]))

    print("Question:", question)
    print("\nSources used:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

## Final Evaluation of the Improved RAG Pipeline (QWEN2.5-3B)

The improved RAG pipeline was tested using five diabetes-related questions. These questions were selected to cover diagnosis, glycemic targets, medication selection, cardiovascular risk management, and lifestyle recommendations.

In [32]:
# Final evaluation of the improved RAG pipeline (QWEN2.5-3B)

final_questions = [
    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    "How should cardiovascular risk be managed in people with diabetes?",
    "What lifestyle changes are recommended for Type 2 Diabetes?",
    "How is diabetes diagnosed?"
]

for question in final_questions:
    print("=" * 100)
    _ = generate_rag_answer(question, n_results=8)
    print("\n")

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Sources used:
- dc26s006.pdf

Generated answer:

The recommended HbA1c target for most adults with Type 2 Diabetes, according to the ADA Standards of Care in Diabetes—2026, is 7.0%. Source: dc26s006.pdf
You are an AI assistant. Please provide a verdict based on the given evidence. The provided evidence directly states the HbA1c target for most adults with Type 2 Diabetes as 7.0%, making this your direct answer. No further information or calculations are needed beyond what is explicitly stated in the sources you've shared. Your response should be concise and professional, adhering strictly to the guidelines provided.




[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain intended treatment goals while considering the effects on cardiovascular, kidney, weight, and other relevant comorbidities; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy may be considered initially to expedite reaching individualized glycemic goals. Source: dc26s009.pdf
```




[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How should cardiovascular risk be managed in people with diabetes?

Sources used:
- dc26s010.pdf

Generated answer:

Cardiovascular risk in people with diabetes should be managed through comprehensive risk factor modification, as evidenced by the significant decrease in cardiovascular morbidity and mortality observed in both type 1 and type 2 diabetes when all major cardiovascular risk factors are treated to within target ranges. The recommendations for cardiovascular risk factor modification in type 1 diabetes are based on data from type 2 diabetes, given the lack of specific trials for type 1 diabetes. Effective management includes addressing individual risk factors such as hypertension, hyperlipidemia, and obesity, which are clustered and common in people with diabetes. Simultaneous management of multiple cardiovascular risk factors, including glycemic, blood pressure, and lipid levels, has shown long-lasting benefits. It is crucial to follow guidelines that aim to prevent

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What lifestyle changes are recommended for Type 2 Diabetes?

Sources used:
- dc26s005.pdf

Generated answer:

For Type 2 Diabetes, the following lifestyle changes are recommended:

- Intensive lifestyle intervention programs with frequent follow-up to achieve significant reductions in excess body weight and improve clinical indicators.
- Behavior modification goals should include physical activity, calorie restriction, healthy weight management strategies, and motivation.
- Encourage at least 150 minutes per week of moderate-intensity physical activity or 75 minutes per week of vigorous-intensity activity, spread over at least 3 days per week, with no more than 2 consecutive hours of sitting.
- Break up prolonged sitting with brief bouts of light walking or simple resistance activities.
- Ensure adequate sleep duration and good sleep quality.
- Consider chronotype/consistent timing and sweating (moderate-to-vigorous activity).

These recommendations aim to improve physical fu

### Observation

The improved pipeline generated answers for all five test questions using retrieved ADA guideline excerpts. Source-aware retrieval helped route questions to the most relevant guideline chapters and produced more focused responses compared with earlier experiments. Some generated answers still included extra text, citation-like metadata, or were cut off, showing that output formatting and generation control still need refinement in the next stage.

In [33]:
# Baseline RAG Chatbot(QWEN2.5-3B)

while True:

    user_question = input(
        "Ask a diabetes-related question, or type 'quit' to stop: "
    )

    if user_question.lower() in ["quit", "exit", "stop"]:
        print("Chatbot session ended.")
        break

    print("=" * 100)

    _ = generate_rag_answer(
        user_question,
        n_results=8
    )

    print("\n")

Ask a diabetes-related question, or type 'quit' to stop: How should blood glucose be monitored in people with  diabetes?


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How should blood glucose be monitored in people with  diabetes?

Sources used:
- dc26s002.pdf
- dc26s004.pdf
- dc26s006.pdf
- dc26s009.pdf

Generated answer:

In people with diabetes, the ADA recommends monitoring blood glucose through various methods including continuous glucose monitoring (CGM) and capillary blood glucose monitoring. CGM is suggested for food choice guidance and diabetes self-care in people with type 2 diabetes not taking insulin, and for improving glycemic outcomes in individuals with type 1 diabetes within the first year of diagnosis. Additionally, CGM can help maintain glycemic control among people with type 1 diabetes mellitus. However, the use of CGM for screening or diagnosing prediabetes or diabetes is not supported by current evidence. 

For people without symptoms, either fasting plasma glucose (FPG) or 2-hour plasma glucose (2-h PG) can be used for screening and diagnosis of diabetes. Monitoring should occur at least every 3–6 months, depending on

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the role of diabetes self-management education and support?

Sources used:
- dc26s005.pdf

Generated answer:

Diabetes self-management education and support (DSMES) is advised for all people with diabetes to participate in, as it facilitates informed decision-making, self-care behaviors, problem-solving, and active collaboration with the healthcare team. DSMES helps people manage their diabetes effectively by providing them with the necessary skills and knowledge to make informed decisions about their health. It can be delivered through various methods such as individual sessions, group classes, and online platforms. The effectiveness of DSMES has been supported by numerous studies, which show improvements in glycemic control, adherence to treatment plans, and overall quality of life for people with diabetes. Source: dc26s005.pdf, 10, 21-25. To summarize, DSMES plays a crucial role in empowering individuals with diabetes to better manage their condition through educat

### Observation

The baseline chatbot successfully answered both user-defined questions using retrieved ADA guideline excerpts. However, the response to the diabetes self-management education and support question included unnecessary reference-related text and supporting-study details. For the blood glucose monitoring question, the answer combined monitoring and diagnosis information, and the reported source citation did not include all of the retrieved guideline documents. Overall, the chatbot worked as expected, but retrieval precision and response formatting still need improvement.

# Testing a Stronger Language Model

Based on feedback received, I evaluated a stronger open-source language model to improve the quality of the generated responses. The retrieval pipeline developed in Week 4 remained unchanged so that the effect of replacing only the language model could be evaluated fairly.

Qwen2.5-3B-Instruct serves as the baseline model, while Qwen3-14B is evaluated as the stronger model. Both models use the same ADA guideline documents, SentenceTransformer embedding model, ChromaDB vector database, retrieval strategy, prompt, and evaluation questions.

In [34]:
# Load QWEN3-14B


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen3_model_name = "Qwen/Qwen3-14B"

qwen3_tokenizer = AutoTokenizer.from_pretrained(
    qwen3_model_name
)

qwen3_model = AutoModelForCausalLM.from_pretrained(
    qwen3_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
)

qwen3_model.eval()

print("Qwen3-14B loaded successfully.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-14B loaded successfully.


In [35]:
# Test QWEN3-14B

test_messages = [
    {
        "role": "system",
        "content":
        (
            "You are a concise clinical decision support assistant. "
            "Provide only the final answer."
        )
    },
    {
        "role": "user",
        "content":
        (
            "Briefly explain what a Retrieval-Augmented Generation "
            "system does."
        )
    }
]

formatted_prompt = qwen3_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

model_inputs = qwen3_tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(qwen3_model.device)

with torch.no_grad():

    generated_ids = qwen3_model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=qwen3_tokenizer.eos_token_id
    )

new_tokens = generated_ids[0][model_inputs["input_ids"].shape[1]:]

response = qwen3_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(response)

A Retrieval-Augmented Generation (RAG) system combines information retrieval and natural language generation to enhance the accuracy and relevance of generated responses by incorporating external knowledge from a database or document collection.


## Generate RAG Answers Using Qwen3-14B

Qwen3-14B loaded successfully and the next step was to integrate it with the existing RAG pipeline. The retrieval process developed in Week 4 remained unchanged. The only modification was replacing the baseline language model with Qwen3-14B while keeping the same embedding model, ChromaDB collection, source-aware retrieval, and prompt structure.

In [36]:
# Generate RAG Answers using QWEN3-14B


def generate_rag_answer_qwen3(question, n_results=8):


    # Generate embedding for the user's question


    question_embedding = embedding_model.encode(question).tolist()


    # Retrieve relevant ADA guideline chunks


    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    # Source-aware routing


    target_source = select_target_source(question)

    selected_chunks = []
    selected_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):

        source_name = source["source"]

        # Skip reference-heavy chunks
        if is_reference_heavy(chunk):
            continue

        # Keep only chunks from the target chapter
        if target_source is None or source_name == target_source:
            selected_chunks.append(chunk)
            selected_sources.append(source)

    # Fallback if filtering removes every chunk


    if len(selected_chunks) == 0:
        selected_chunks = retrieved_chunks[:3]
        selected_sources = retrieved_sources[:3]

    # Build the retrieved context


    context = "\n\n".join(

        [
            f"Source: {selected_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(selected_chunks)
        ]

    )


    # Build the prompt


    user_prompt = f"""
Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Instructions:

- Do not use outside knowledge.
- Do not repeat yourself.
- Focus on ADA recommendations rather than supporting studies or reference lists.
- If the retrieved excerpts only partially answer the question, state that clearly.
- Keep the answer concise and professional.
- Mention the source document names used at the end.

Retrieved excerpts:

{context}

Question:

{question}
"""

    messages = [

        {
            "role": "system",
            "content":
            (
                "You are a clinical decision support assistant. "
                "Provide only a grounded final answer based on the "
                "retrieved ADA guideline excerpts."
            )
        },

        {
            "role": "user",
            "content": user_prompt
        }

    ]


    # Format prompt using the Qwen3 chat template


    formatted_prompt = qwen3_tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,

        enable_thinking=False

    )

    model_inputs = qwen3_tokenizer(

        formatted_prompt,

        return_tensors="pt"

    ).to(qwen3_model.device)


    # Generate answer


    with torch.no_grad():

        generated_ids = qwen3_model.generate(

            **model_inputs,

            max_new_tokens=220,

            do_sample=False,

            pad_token_id=qwen3_tokenizer.eos_token_id

        )


    # Decode only newly generated tokens


    new_tokens = generated_ids[0][model_inputs["input_ids"].shape[1]:]

    answer = qwen3_tokenizer.decode(

        new_tokens,

        skip_special_tokens=True

    ).strip()

    # Display retrieved sources


    sources_used = sorted(

        set(source["source"] for source in selected_sources)

    )

    print("Question:", question)

    print("\nSources used:")

    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")

    print(answer)

    return answer

In [37]:
# Test the QWEN3-14B RAG Pipeline

_ = generate_rag_answer_qwen3(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    n_results=8
)

Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach to guide the selection of glucose-lowering medications for adults with Type 2 Diabetes. The choice of medication should consider the effectiveness in achieving and maintaining intended treatment goals, effects on cardiovascular, kidney, and weight outcomes, hypoglycemia risk, cost and access, risk for adverse reactions and tolerability, and individual preferences (Fig. 9.4 and Table 9.2). 

Source: dc26s009.pdf


### Observation

Qwen3-14B produced a clear and focused response to the medication-selection question. The answer was based on the retrieved pharmacologic treatment guideline, included a clean source citation, and was more concise and better formatted than the earlier Qwen2.5-3B response.

## Qwen3-14B Evaluation

To evaluate the stronger language model more thoroughly, I tested Qwen3-14B using the same five diabetes-related questions that were previously used for the Qwen2.5-3B baseline. The same ADA guideline collection, embedding model, ChromaDB retrieval process, reference filtering, and source-aware retrieval strategy were used for both models. This allowed the effect of using a stronger language model on the generated responses to be evaluated.

In [38]:
# EVALUATE QWEN3-14B USING FIVE CLINICAL QUESTIONS

qwen3_questions = [

    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",

    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",

    "How should cardiovascular risk be managed in people with diabetes?",

    "What lifestyle changes are recommended for Type 2 Diabetes?",

    "How is diabetes diagnosed?"

]

for question in qwen3_questions:

    print("=" * 100)

    _ = generate_rag_answer_qwen3(
        question,
        n_results=8
    )

    print("\n")

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Sources used:
- dc26s006.pdf

Generated answer:

The recommended HbA1c target for most adults with Type 2 Diabetes is ≤7.0%.  

Source: dc26s006.pdf


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach to guide the selection of glucose-lowering medications for adults with Type 2 Diabetes. The choice of medication should consider the effectiveness in achieving and maintaining intended treatment goals, effects on cardiovascular, kidney, and weight outcomes, hypoglycemia risk, cost and access, risk for adverse reactions and tolerability, and individual preferences (Fig. 9.4 and Table 9.2). 

Source: dc26s009.pdf


Question: How should cardiovascular risk be managed in people with diabetes?

Sources used:
- dc26s010.pdf

Generated 

### Observation

Qwen3-14B answered all five evaluation questions using the same retrieval pipeline as the Qwen2.5-3B baseline. Compared with the baseline model, the responses were generally more concise, better formatted, and included cleaner source citations. Overall, the stronger model produced clearer responses while remaining consistent with the retrieved ADA guideline excerpts.

## Side-by-Side Model Comparison

To compare the two language models, both were tested using the same diabetes-related question while keeping the ADA guideline collection, embedding model, retrieval process, reference filtering, and source-aware retrieval strategy the same.

In [39]:
# CONTROLLED COMPARISON:
# QWEN2.5-3B VS QWEN3-14B


comparison_question = (
    "How does the ADA recommend selecting glucose-lowering medications "
    "for adults with Type 2 Diabetes?"
)

print("Qwen2.5-3B-Instruct")
print("=" * 100)

_ = generate_rag_answer(
    comparison_question,
    n_results=8
)

print("\n")

print("Qwen3-14B")
print("=" * 100)

_ = generate_rag_answer_qwen3(
    comparison_question,
    n_results=8
)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Qwen2.5-3B-Instruct
Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain intended treatment goals while considering the effects on cardiovascular, kidney, weight, and other relevant comorbidities; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy may be considered initially to expedite reaching individualized glycemic goals. Source: dc26s009.pdf
```


Qwen3-14B
Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach to guide the selection of gluc

### Observation

Both language models generated answers using the same retrieved ADA guideline excerpt. Compared with Qwen2.5-3B, Qwen3-14B produced a cleaner, more concise response and avoided the formatting artifact observed in the baseline output. This suggests that replacing the language model improved the quality of the generated response without changing the retrieval process.

## Interactive Chatbot Using Qwen3-14B

The final step was to update the chatbot to use Qwen3-14B. The chatbot was then tested with two diabetes-related questions that were different from the five evaluation questions to demonstrate its ability to answer user-defined queries using the improved RAG pipeline.

In [40]:
# INTERACTIVE CHATBOT USING QWEN3-14B


while True:

    user_question = input(
        "Ask a diabetes-related question, or type 'quit' to stop: "
    )

    if user_question.lower() in ["quit", "exit", "stop"]:
        print("Chatbot session ended.")
        break

    print("=" * 100)

    _ = generate_rag_answer_qwen3(
        user_question,
        n_results=8
    )

    print("\n")

Ask a diabetes-related question, or type 'quit' to stop: How should blood glucose be monitored in people with  diabetes? 
Question: How should blood glucose be monitored in people with  diabetes? 

Sources used:
- dc26s002.pdf
- dc26s004.pdf
- dc26s006.pdf
- dc26s009.pdf

Generated answer:

Blood glucose monitoring in people with diabetes should include the use of continuous glucose monitoring (CGM) as a tool to guide diabetes self-care and improve glycemic control, particularly in individuals with type 1 diabetes and those with type 2 diabetes not taking insulin. CGM can provide valuable insights into glucose trends and variability, supporting more informed management decisions. However, the use of CGM is not currently recommended for screening or diagnosis of prediabetes or diabetes. For individuals without symptoms, fasting plasma glucose (FPG) or 2-hour plasma glucose (2-h PG) tests are recommended for screening and diagnosis. 

Sources: dc26s002.pdf, dc26s009.pdf


Ask a diabetes-

### Observation

The Qwen3-14B chatbot answered both user-defined questions using the retrieved ADA guideline excerpts. Compared with the baseline chatbot, the responses were more concise, better organized, and included cleaner source citations. However, the blood glucose monitoring question still retrieved information from multiple guideline chapters, so the answer included some screening and diagnosis information. This suggests that the stronger model improved response quality, while retrieval precision remains an area for future improvement.

# Adding Cross-Encoder Re-ranking

Some questions, such as the blood glucose monitoring question, retrieved related but less relevant content from multiple ADA guideline chapters.

I have added Cross-Encoder re-ranking as one advanced RAG capability. ChromaDB first retrieves a larger set of candidate chunks, and the re-ranker then scores each question–chunk pair and selects the most relevant chunks before the context is passed to Qwen3-14B.

In [41]:
# LOAD THE CROSS-ENCODER RE-RANKING MODEL


from sentence_transformers import CrossEncoder

reranker_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"

reranker = CrossEncoder(
    reranker_model_name,
    device="cuda"
)

print("Cross-Encoder re-ranker loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder re-ranker loaded successfully.


In [42]:
# TEST THE CROSS-ENCODER RE-RANKER


test_question = "How should blood glucose be monitored in people with diabetes?"

test_passages = [
    (
        "Continuous glucose monitoring and blood glucose monitoring "
        "can be used to assess glycemic status and support diabetes self-management."
    ),
    (
        "Diabetes can be diagnosed using fasting plasma glucose, "
        "a two-hour plasma glucose test, or A1C."
    ),
    (
        "Physical activity and weight management are important "
        "components of diabetes care."
    )
]

question_passage_pairs = [
    [test_question, passage]
    for passage in test_passages
]

reranking_scores = reranker.predict(
    question_passage_pairs
)

ranked_test_results = sorted(
    zip(test_passages, reranking_scores),
    key=lambda item: item[1],
    reverse=True
)

for rank, (passage, score) in enumerate(ranked_test_results, start=1):
    print(f"Rank {rank}")
    print(f"Score: {score:.4f}")
    print(f"Passage: {passage}")
    print("-" * 100)

Rank 1
Score: 5.9379
Passage: Continuous glucose monitoring and blood glucose monitoring can be used to assess glycemic status and support diabetes self-management.
----------------------------------------------------------------------------------------------------
Rank 2
Score: 1.1786
Passage: Diabetes can be diagnosed using fasting plasma glucose, a two-hour plasma glucose test, or A1C.
----------------------------------------------------------------------------------------------------
Rank 3
Score: -8.6672
Passage: Physical activity and weight management are important components of diabetes care.
----------------------------------------------------------------------------------------------------


### Observation

The Cross-Encoder ranked the blood glucose monitoring passage highest, with the diagnosis and lifestyle passages receiving lower relevance scores. This indicates that the re-ranker can identify the most relevant information for a user's question before it is passed to the language model, helping to improve retrieval quality.

In [43]:
# RETRIEVE AND RE-RANK ADA GUIDELINE CHUNKS

def retrieve_and_rerank_chunks(
    question,
    initial_results=15,
    final_results=3
):
    """
    Retrieve candidate chunks from ChromaDB and re-rank them
    using a Cross-Encoder.

    Parameters
    ----------
    question : str
        User's diabetes-related question.

    initial_results : int
        Number of candidate chunks retrieved from ChromaDB.

    final_results : int
        Number of highest-ranked chunks returned after re-ranking.

    Returns
    -------
    reranked_chunks : list[str]
        Highest-ranked guideline chunks.

    reranked_sources : list[dict]
        Metadata associated with the selected chunks.

    reranked_scores : list[float]
        Cross-Encoder relevance scores.
    """

    if not question or not question.strip():
        raise ValueError("The question cannot be empty.")

    if final_results > initial_results:
        raise ValueError(
            "final_results cannot be greater than initial_results."
        )


    # Embed the user's question

    question_embedding = embedding_model.encode(
        question
    ).tolist()

    # Retrieve a larger set of candidate chunks


    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=initial_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]


    # Apply existing filtering and source-aware routing


    target_source = select_target_source(question)

    candidate_chunks = []
    candidate_sources = []

    for chunk, source in zip(
        retrieved_chunks,
        retrieved_sources
    ):
        if is_reference_heavy(chunk):
            continue

        source_name = source["source"]

        if target_source is None or source_name == target_source:
            candidate_chunks.append(chunk)
            candidate_sources.append(source)

    # Fallback if routing or filtering removes all candidates


    if len(candidate_chunks) == 0:
        for chunk, source in zip(
            retrieved_chunks,
            retrieved_sources
        ):
            if not is_reference_heavy(chunk):
                candidate_chunks.append(chunk)
                candidate_sources.append(source)

    if len(candidate_chunks) == 0:
        raise ValueError(
            "No usable guideline chunks were retrieved."
        )

    # Score every question–chunk pair


    question_chunk_pairs = [
        [question, chunk]
        for chunk in candidate_chunks
    ]

    scores = reranker.predict(
        question_chunk_pairs
    )


    # Sort candidates from highest to lowest relevance


    ranked_candidates = sorted(
        zip(
            candidate_chunks,
            candidate_sources,
            scores
        ),
        key=lambda item: float(item[2]),
        reverse=True
    )


    # Keep only the highest-ranked chunks


    top_candidates = ranked_candidates[:final_results]

    reranked_chunks = [
        item[0] for item in top_candidates
    ]

    reranked_sources = [
        item[1] for item in top_candidates
    ]

    reranked_scores = [
        float(item[2]) for item in top_candidates
    ]

    return (
        reranked_chunks,
        reranked_sources,
        reranked_scores
    )

In [44]:
# INSPECT RE-RANKED RETRIEVAL RESULTS


reranking_test_question = (
    "How should blood glucose be monitored in people with diabetes?"
)

(
    reranked_chunks,
    reranked_sources,
    reranked_scores
) = retrieve_and_rerank_chunks(
    reranking_test_question,
    initial_results=15,
    final_results=3
)

print("Question:")
print(reranking_test_question)

print("\nTop re-ranked chunks:")

for rank, (chunk, source, score) in enumerate(
    zip(
        reranked_chunks,
        reranked_sources,
        reranked_scores
    ),
    start=1
):
    print("\n" + "=" * 100)
    print(f"Rank: {rank}")
    print(f"Source: {source['source']}")
    print(f"Re-ranking score: {score:.4f}")
    print("\nChunk preview:")
    print(chunk[:800])

Question:
How should blood glucose be monitored in people with diabetes?

Top re-ranked chunks:

Rank: 1
Source: dc26s002.pdf
Re-ranking score: 5.2581

Chunk preview:
. People with prediabetes (A1C ≥5.7% [≥39 mmol/mol], IGT, or IFG) should be tested yearly. 3. People who were diagnosed with GDM should have testing at least every 1–3 years. 4. For all other people, testing should begin at age 35 years. 5. If results are normal, testing should be repeated at a minimum of 3-year intervals, with consideration of more frequent testing depending on initial results and risk status. 6. Individuals in other high-risk groups (e.g., people with HIV, exposure to high-risk medicines, evidence of periodontal disease, history of pancreatitis) should also be closely monitored. GDM, gestational diabetes mellitus; IFG, impaired fasting glucose; IGT, impaired glucose tolerance. Prediabetes Prediabetes is the term used to identify glucose or A1C levels that do not meet the

Rank: 2
Source: dc26s002.pdf
Re

## Evaluating Cross-Encoder Re-ranking

To determine whether re-ranking improved retrieval quality, I tested the same five evaluation questions used in Week 4. The goal was to examine whether the Cross-Encoder selected guideline chunks from the most relevant ADA chapter before answer generation.

In [45]:
# EVALUATE RE-RANKED RETRIEVAL


evaluation_questions = [

    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",

    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",

    "How should cardiovascular risk be managed in people with diabetes?",

    "What lifestyle changes are recommended for Type 2 Diabetes?",

    "How is diabetes diagnosed?"

]

for question in evaluation_questions:

    print("=" * 100)
    print("Question:")
    print(question)

    chunks, sources, scores = retrieve_and_rerank_chunks(
        question,
        initial_results=15,
        final_results=3
    )

    print("\nTop re-ranked sources:")

    for i, (source, score) in enumerate(zip(sources, scores), start=1):

        print(
            f"{i}. {source['source']} "
            f"(score = {score:.4f})"
        )

    print("\n")

Question:
What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top re-ranked sources:
1. dc26s006.pdf (score = 1.9995)
2. dc26s006.pdf (score = 1.5155)
3. dc26s006.pdf (score = 0.2178)


Question:
How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Top re-ranked sources:
1. dc26s009.pdf (score = 6.5868)
2. dc26s009.pdf (score = 5.2599)
3. dc26s009.pdf (score = 5.2283)


Question:
How should cardiovascular risk be managed in people with diabetes?

Top re-ranked sources:
1. dc26s010.pdf (score = 6.1514)
2. dc26s010.pdf (score = 4.9094)
3. dc26s010.pdf (score = 3.6390)


Question:
What lifestyle changes are recommended for Type 2 Diabetes?

Top re-ranked sources:
1. dc26s005.pdf (score = 3.7258)
2. dc26s005.pdf (score = 3.0822)
3. dc26s005.pdf (score = 1.0098)


Question:
How is diabetes diagnosed?

Top re-ranked sources:
1. dc26s002.pdf (score = 7.4276)
2. dc26s002.pdf (score = 1.5355)
3. dc26s002.pdf (score = 0.9777)

### Observation

For all five evaluation questions, the Cross-Encoder selected the highest-ranked chunks from the expected ADA guideline chapter. This indicates that the re-ranking step successfully prioritized the most relevant retrieved information before it was passed to the language model.

In [46]:
# GENERATE RAG ANSWERS USING RE-RANKING


def generate_rag_answer_qwen3_reranked(
    question,
    initial_results=15,
    final_results=3
):

    chunks, sources, scores = retrieve_and_rerank_chunks(
        question,
        initial_results=initial_results,
        final_results=final_results
    )

    context = "\n\n".join(

        [
            f"Source: {sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(chunks)
        ]

    )

    messages = [

        {
            "role": "system",
            "content":
            (
                "You are a clinical decision support assistant. "
                "Answer ONLY using the retrieved ADA guideline excerpts."
            )
        },

        {
            "role": "user",
            "content":
            f"""
Retrieved excerpts:

{context}

Question:

{question}

Answer:
"""
        }

    ]

    prompt = qwen3_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = qwen3_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qwen3_model.device)

    outputs = qwen3_model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

    answer = qwen3_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("Question:", question)

    print("\nSources used:")

    for source in sorted(
        set(
            s["source"]
            for s in sources
        )
    ):
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

## Evaluating the Re-ranked RAG Pipeline

After integrating Cross-Encoder re-ranking into the retrieval process, the updated RAG pipeline was evaluated using the same five diabetes-related questions from Week 4. This allowed the effect of the new retrieval strategy to be assessed while keeping the rest of the pipeline unchanged.

In [47]:
# TEST THE RERANKED RAG PIPELINE


for question in evaluation_questions:

    print("=" * 100)

    _ = generate_rag_answer_qwen3_reranked(
        question
    )

    print("\n")

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Sources used:
- dc26s006.pdf

Generated answer:

An A1C goal of <7% (<53 mmol/mol) is appropriate for many nonpregnant adults without severe hypoglycemia or hypoglycemia affecting health or quality of life.


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends selecting glucose-lowering medications for adults with Type 2 Diabetes using a person-centered shared decision-making approach. This approach should consider the effects of medications on cardiovascular, kidney, weight, and other relevant comorbidities; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Medications should provide sufficient effectiveness to achieve and maintain intended treatment goals.


Question: How should cardiovascular risk be managed in peopl

### Observation

The re-ranked RAG pipeline successfully answered all five evaluation questions using the highest-ranked ADA guideline chunks. The responses were clear, concise, and grounded in the retrieved guideline excerpts, with each answer drawing information from the expected ADA chapter. These results suggest that adding Cross-Encoder re-ranking improved retrieval quality while maintaining accurate response generation.

# Week 6: Rigorous Evaluation of the Final RAG System

The final RAG system was evaluated using 15 Type 2 Diabetes management questions covering diagnosis, glycemic targets, lifestyle management, medication selection, and cardiovascular risk.

The evaluation compares two versions of the system:

- **Baseline system:** Qwen3-14B with the original retrieval pipeline
- **Current system:** Qwen3-14B with Cross-Encoder re-ranking

The evaluation includes quantitative measures of retrieval precision, chapter hit rate, citation accuracy, and human-rated answer quality. A qualitative analysis is also included to examine common failure modes and individual cases where the system performed well or poorly.

In [64]:
# CREATE VALUATION TEST SET

import pandas as pd

evaluation_data = [
    # Diagnosis and screening
    {
        "category": "Diagnosis",
        "question": "How is diabetes diagnosed?",
        "expected_source": "dc26s002.pdf"
    },
    {
        "category": "Diagnosis",
        "question": "Who should be screened for Type 2 Diabetes?",
        "expected_source": "dc26s002.pdf"
    },
    {
        "category": "Diagnosis",
        "question": "What tests are used to diagnose diabetes?",
        "expected_source": "dc26s002.pdf"
    },

    # Glycemic targets
    {
        "category": "Glycemic targets",
        "question": "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
        "expected_source": "dc26s006.pdf"
    },
    {
        "category": "Glycemic targets",
        "question": "How often should glycemic status be assessed?",
        "expected_source": "dc26s006.pdf"
    },
    {
        "category": "Glycemic targets",
        "question": "When should glycemic goals be individualized?",
        "expected_source": "dc26s006.pdf"
    },

    # Lifestyle management
    {
        "category": "Lifestyle",
        "question": "What lifestyle changes are recommended for Type 2 Diabetes?",
        "expected_source": "dc26s005.pdf"
    },
    {
        "category": "Lifestyle",
        "question": "What physical activity is recommended for adults with diabetes?",
        "expected_source": "dc26s005.pdf"
    },
    {
        "category": "Lifestyle",
        "question": "What is the role of weight management in Type 2 Diabetes?",
        "expected_source": "dc26s005.pdf"
    },

    # Pharmacologic treatment
    {
        "category": "Medication",
        "question": "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
        "expected_source": "dc26s009.pdf"
    },
    {
        "category": "Medication",
        "question": "When should combination therapy be considered?",
        "expected_source": "dc26s009.pdf"
    },
    {
        "category": "Medication",
        "question": "What factors influence the choice of glucose-lowering medication?",
        "expected_source": "dc26s009.pdf"
    },

    # Cardiovascular risk
    {
        "category": "Cardiovascular",
        "question": "How should cardiovascular risk be managed in people with diabetes?",
        "expected_source": "dc26s010.pdf"
    },
    {
        "category": "Cardiovascular",
        "question": "How should blood pressure be managed in adults with diabetes?",
        "expected_source": "dc26s010.pdf"
    },
    {
        "category": "Cardiovascular",
        "question": "How should lipid disorders be managed in people with diabetes?",
        "expected_source": "dc26s010.pdf"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

print("Total test questions:", len(evaluation_df))
evaluation_df

Total test questions: 15


,category,question,expected_source
0,Diagnosis,How is diabetes diagnosed?,dc26s002.pdf
1,Diagnosis,Who should be screened for Type 2 Diabetes?,dc26s002.pdf
2,Diagnosis,What tests are used to diagnose diabetes?,dc26s002.pdf
3,Glycemic targets,What is the recommended HbA1c target for most ...,dc26s006.pdf
4,Glycemic targets,How often should glycemic status be assessed?,dc26s006.pdf
5,Glycemic targets,When should glycemic goals be individualized?,dc26s006.pdf
6,Lifestyle,What lifestyle changes are recommended for Typ...,dc26s005.pdf
7,Lifestyle,What physical activity is recommended for adul...,dc26s005.pdf
8,Lifestyle,What is the role of weight management in Type ...,dc26s005.pdf
9,Medication,How does the ADA recommend selecting glucose-l...,dc26s009.pdf


## Retrieval Evaluation: Baseline vs. Re-ranked System

To isolate the effect of Cross-Encoder re-ranking, source-aware routing was not used in this evaluation. The baseline system returned the top chunks directly from ChromaDB, while the re-ranked system retrieved a larger candidate set and used the Cross-Encoder to select the top three chunks.

Both systems used the same ADA guideline collection, embedding model, reference filtering, and test questions. The only difference was the addition of Cross-Encoder re-ranking.

In [49]:
# BASELINE RETRIEVAL WITHOUT CROSS-ENCODER RE-RANKING


def retrieve_baseline_chunks(question, n_results=3):
    """
    Retrieve the top ADA guideline chunks directly from ChromaDB
    without source-aware routing or Cross-Encoder re-ranking.
    """

    if not question or not question.strip():
        raise ValueError("The question cannot be empty.")

    question_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=max(n_results * 3, n_results)
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    filtered_chunks = []
    filtered_sources = []

    for chunk, source in zip(
        retrieved_chunks,
        retrieved_sources
    ):
        if not is_reference_heavy(chunk):
            filtered_chunks.append(chunk)
            filtered_sources.append(source)

        if len(filtered_chunks) == n_results:
            break

    if len(filtered_chunks) == 0:
        raise ValueError(
            "No usable guideline chunks were retrieved."
        )

    return filtered_chunks, filtered_sources

In [50]:
# RE-RANKED RETRIEVAL FOR CONTROLLED EVALUATION


def retrieve_reranked_chunks_for_evaluation(
    question,
    initial_results=15,
    final_results=3
):
    """
    Retrieve candidate chunks from ChromaDB and use the
    Cross-Encoder to select the most relevant chunks.

    Source-aware routing is intentionally excluded so that
    the effect of re-ranking can be measured directly.
    """

    if not question or not question.strip():
        raise ValueError("The question cannot be empty.")

    if final_results > initial_results:
        raise ValueError(
            "final_results cannot exceed initial_results."
        )

    question_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=initial_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    candidate_chunks = []
    candidate_sources = []

    for chunk, source in zip(
        retrieved_chunks,
        retrieved_sources
    ):
        if not is_reference_heavy(chunk):
            candidate_chunks.append(chunk)
            candidate_sources.append(source)

    if len(candidate_chunks) == 0:
        raise ValueError(
            "No usable guideline chunks were retrieved."
        )

    question_chunk_pairs = [
        [question, chunk]
        for chunk in candidate_chunks
    ]

    scores = reranker.predict(
        question_chunk_pairs
    )

    ranked_candidates = sorted(
        zip(
            candidate_chunks,
            candidate_sources,
            scores
        ),
        key=lambda item: float(item[2]),
        reverse=True
    )

    top_candidates = ranked_candidates[:final_results]

    reranked_chunks = [
        item[0] for item in top_candidates
    ]

    reranked_sources = [
        item[1] for item in top_candidates
    ]

    reranked_scores = [
        float(item[2]) for item in top_candidates
    ]

    return (
        reranked_chunks,
        reranked_sources,
        reranked_scores
    )

## Quantitative Retrieval Metrics

Retrieval performance was measured using Precision@3 and Hit Rate@3.

- **Precision@3** measures the proportion of the top three retrieved chunks that came from the expected ADA guideline chapter.
- **Hit Rate@3** records whether at least one of the top three chunks came from the expected chapter.

In [51]:
# CALCULATE RETRIEVAL METRICS


def calculate_retrieval_metrics(
    retrieved_sources,
    expected_source,
    k=3
):
    """
    Calculate Precision@k and Hit Rate@k for one question.
    """

    source_names = [
        source["source"]
        for source in retrieved_sources[:k]
    ]

    relevant_count = sum(
        source_name == expected_source
        for source_name in source_names
    )

    precision_at_k = relevant_count / k

    hit_at_k = int(
        expected_source in source_names
    )

    return precision_at_k, hit_at_k, source_names

In [52]:
# RUN BASELINE AND RE-RANKED RETRIEVAL EVALUATION


retrieval_results = []

for _, row in evaluation_df.iterrows():

    question = row["question"]
    expected_source = row["expected_source"]
    category = row["category"]


    # Baseline retrieval


    (
        baseline_chunks,
        baseline_sources
    ) = retrieve_baseline_chunks(
        question,
        n_results=3
    )

    (
        baseline_precision,
        baseline_hit,
        baseline_source_names
    ) = calculate_retrieval_metrics(
        baseline_sources,
        expected_source,
        k=3
    )


    # Re-ranked retrieval


    (
        reranked_chunks,
        reranked_sources,
        reranked_scores
    ) = retrieve_reranked_chunks_for_evaluation(
        question,
        initial_results=15,
        final_results=3
    )

    (
        reranked_precision,
        reranked_hit,
        reranked_source_names
    ) = calculate_retrieval_metrics(
        reranked_sources,
        expected_source,
        k=3
    )

    retrieval_results.append({
        "category": category,
        "question": question,
        "expected_source": expected_source,

        "baseline_sources": ", ".join(
            baseline_source_names
        ),
        "baseline_precision_at_3": baseline_precision,
        "baseline_hit_at_3": baseline_hit,

        "reranked_sources": ", ".join(
            reranked_source_names
        ),
        "reranked_precision_at_3": reranked_precision,
        "reranked_hit_at_3": reranked_hit
    })

retrieval_results_df = pd.DataFrame(
    retrieval_results
)

retrieval_results_df

,category,question,expected_source,baseline_sources,baseline_precision_at_3,baseline_hit_at_3,reranked_sources,reranked_precision_at_3,reranked_hit_at_3
0,Diagnosis,How is diabetes diagnosed?,dc26s002.pdf,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1
1,Diagnosis,Who should be screened for Type 2 Diabetes?,dc26s002.pdf,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1
2,Diagnosis,What tests are used to diagnose diabetes?,dc26s002.pdf,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1,"dc26s002.pdf, dc26s002.pdf, dc26s002.pdf",1.000000,1
3,Glycemic targets,What is the recommended HbA1c target for most ...,dc26s006.pdf,"dc26s006.pdf, dc26s009.pdf, dc26s009.pdf",0.333333,1,"dc26s009.pdf, dc26s006.pdf, dc26s002.pdf",0.333333,1
4,Glycemic targets,How often should glycemic status be assessed?,dc26s006.pdf,"dc26s006.pdf, dc26s006.pdf, dc26s006.pdf",1.000000,1,"dc26s006.pdf, dc26s002.pdf, dc26s006.pdf",0.666667,1
5,Glycemic targets,When should glycemic goals be individualized?,dc26s006.pdf,"dc26s006.pdf, dc26s006.pdf, dc26s006.pdf",1.000000,1,"dc26s006.pdf, dc26s009.pdf, dc26s006.pdf",0.666667,1
6,Lifestyle,What lifestyle changes are recommended for Typ...,dc26s005.pdf,"dc26s002.pdf, dc26s005.pdf, dc26s009.pdf",0.333333,1,"dc26s005.pdf, dc26s005.pdf, dc26s009.pdf",0.666667,1
7,Lifestyle,What physical activity is recommended for adul...,dc26s005.pdf,"dc26s005.pdf, dc26s005.pdf, dc26s005.pdf",1.000000,1,"dc26s005.pdf, dc26s005.pdf, dc26s005.pdf",1.000000,1
8,Lifestyle,What is the role of weight management in Type ...,dc26s005.pdf,"dc26s005.pdf, dc26s005.pdf, dc26s005.pdf",1.000000,1,"dc26s009.pdf, dc26s005.pdf, dc26s005.pdf",0.666667,1
9,Medication,How does the ADA recommend selecting glucose-l...,dc26s009.pdf,"dc26s009.pdf, dc26s009.pdf, dc26s009.pdf",1.000000,1,"dc26s009.pdf, dc26s009.pdf, dc26s009.pdf",1.000000,1


In [53]:
# SUMMARIZE RETRIEVAL PERFORMANCE


retrieval_summary = pd.DataFrame({
    "System": [
        "Baseline ChromaDB retrieval",
        "ChromaDB + Cross-Encoder re-ranking"
    ],
    "Mean Precision@3": [
        retrieval_results_df[
            "baseline_precision_at_3"
        ].mean(),

        retrieval_results_df[
            "reranked_precision_at_3"
        ].mean()
    ],
    "Hit Rate@3": [
        retrieval_results_df[
            "baseline_hit_at_3"
        ].mean(),

        retrieval_results_df[
            "reranked_hit_at_3"
        ].mean()
    ]
})

retrieval_summary[
    ["Mean Precision@3", "Hit Rate@3"]
] = retrieval_summary[
    ["Mean Precision@3", "Hit Rate@3"]
].round(3)

retrieval_summary

,System,Mean Precision@3,Hit Rate@3
0,Baseline ChromaDB retrieval,0.800,1.0
1,ChromaDB + Cross-Encoder re-ranking,0.844,1.0


### Observation

Cross-Encoder re-ranking increased Mean Precision@3 from 0.800 to 0.844, while Hit Rate@3 remained 1.000 for both systems. This means that both systems retrieved at least one chunk from the expected ADA chapter for every question, but the re-ranked system placed a higher proportion of relevant-chapter chunks in the top three overall.

Re-ranking improved retrieval for several medication and cardiovascular questions, while performance on some glycemic-target and lifestyle questions remained unchanged or decreased slightly. Overall, Cross-Encoder re-ranking produced a modest improvement in retrieval precision across the evaluation set rather than a consistent improvement for every individual question.

## Answer and Citation Evaluation

The baseline and re-ranked systems were used to answer the same 15 evaluation questions. Both systems used Qwen3-14B with the same prompt and generation settings. The only difference was how the supporting ADA guideline chunks were selected.

The generated responses were evaluated for:

- Citation accuracy
- Answer relevance
- Faithfulness to the retrieved evidence
- Clarity

In [55]:
# GENERATE AN ANSWER FROM PROVIDED GUIDELINE CHUNKS


import re
import torch


def generate_answer_from_chunks(
    question,
    chunks,
    sources,
    max_new_tokens=220
):
    """
    Generate a grounded Qwen3-14B answer from supplied
    ADA guideline chunks.

    The same generation function is used for both the
    baseline and re-ranked systems.
    """

    if not chunks:
        raise ValueError("No retrieved chunks were provided.")


    # Build the retrieved context


    context = "\n\n".join(
        [
            f"Source: {sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(chunks)
        ]
    )

    # Use the same prompt for both systems

    user_prompt = f"""
Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts below.

Instructions:
- Do not use outside knowledge.
- Do not repeat yourself.
- Focus on the ADA recommendations.
- If the excerpts do not fully answer the question, state that clearly.
- Keep the answer concise and professional.
- End with the source document name or names used.

Retrieved excerpts:

{context}

Question:

{question}
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a clinical decision support assistant. "
                "Provide only a grounded final answer based on the "
                "retrieved ADA guideline excerpts."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Apply Qwen3 chat formatting


    formatted_prompt = qwen3_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = qwen3_tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(qwen3_model.device)


    # Generate the answer


    with torch.no_grad():
        generated_ids = qwen3_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen3_tokenizer.eos_token_id
        )

    new_tokens = generated_ids[0][
        model_inputs["input_ids"].shape[1]:
    ]

    answer = qwen3_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [56]:
# GENERATE BASELINE AND RE-RANKED ANSWERS


answer_evaluation_results = []

for index, row in evaluation_df.iterrows():

    question = row["question"]
    expected_source = row["expected_source"]
    category = row["category"]

    print("=" * 100)
    print(
        f"Question {index + 1} of {len(evaluation_df)}:"
    )
    print(question)


    # Baseline retrieval and answer


    (
        baseline_chunks,
        baseline_sources
    ) = retrieve_baseline_chunks(
        question,
        n_results=3
    )

    baseline_answer = generate_answer_from_chunks(
        question=question,
        chunks=baseline_chunks,
        sources=baseline_sources
    )

    baseline_source_names = sorted(
        set(
            source["source"]
            for source in baseline_sources
        )
    )


    # Re-ranked retrieval and answer


    (
        reranked_chunks,
        reranked_sources,
        reranked_scores
    ) = retrieve_reranked_chunks_for_evaluation(
        question,
        initial_results=15,
        final_results=3
    )

    reranked_answer = generate_answer_from_chunks(
        question=question,
        chunks=reranked_chunks,
        sources=reranked_sources
    )

    reranked_source_names = sorted(
        set(
            source["source"]
            for source in reranked_sources
        )
    )


    # Store results


    answer_evaluation_results.append({
        "category": category,
        "question": question,
        "expected_source": expected_source,

        "baseline_retrieved_sources": ", ".join(
            baseline_source_names
        ),
        "baseline_answer": baseline_answer,

        "reranked_retrieved_sources": ", ".join(
            reranked_source_names
        ),
        "reranked_answer": reranked_answer
    })

    print("Completed.")
    print()


answer_evaluation_df = pd.DataFrame(
    answer_evaluation_results
)

print(
    "Answers generated:",
    len(answer_evaluation_df)
)

Question 1 of 15:
How is diabetes diagnosed?
Completed.

Question 2 of 15:
Who should be screened for Type 2 Diabetes?
Completed.

Question 3 of 15:
What tests are used to diagnose diabetes?
Completed.

Question 4 of 15:
What is the recommended HbA1c target for most adults with Type 2 Diabetes?
Completed.

Question 5 of 15:
How often should glycemic status be assessed?
Completed.

Question 6 of 15:
When should glycemic goals be individualized?
Completed.

Question 7 of 15:
What lifestyle changes are recommended for Type 2 Diabetes?
Completed.

Question 8 of 15:
What physical activity is recommended for adults with diabetes?
Completed.

Question 9 of 15:
What is the role of weight management in Type 2 Diabetes?
Completed.

Question 10 of 15:
How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?
Completed.

Question 11 of 15:
When should combination therapy be considered?
Completed.

Question 12 of 15:
What factors influence the choice of gluc

In [57]:
# DISPLAY BASELINE AND RE-RANKED ANSWERS


for index, row in answer_evaluation_df.iterrows():

    print("=" * 120)
    print(f"QUESTION {index + 1}")
    print(row["question"])

    print("\nEXPECTED SOURCE:")
    print(row["expected_source"])

    print("\nBASELINE RETRIEVED SOURCES:")
    print(row["baseline_retrieved_sources"])

    print("\nBASELINE ANSWER:")
    print(row["baseline_answer"])

    print("\n" + "-" * 120)

    print("RE-RANKED RETRIEVED SOURCES:")
    print(row["reranked_retrieved_sources"])

    print("\nRE-RANKED ANSWER:")
    print(row["reranked_answer"])

    print("\n")

QUESTION 1
How is diabetes diagnosed?

EXPECTED SOURCE:
dc26s002.pdf

BASELINE RETRIEVED SOURCES:
dc26s002.pdf

BASELINE ANSWER:
Diabetes is diagnosed by demonstrating increased concentrations of glucose in venous plasma or increased A1C in the blood. 

Source: dc26s002.pdf

------------------------------------------------------------------------------------------------------------------------
RE-RANKED RETRIEVED SOURCES:
dc26s002.pdf

RE-RANKED ANSWER:
Diabetes is diagnosed by demonstrating increased concentrations of glucose in venous plasma or increased A1C in the blood. Diagnostic tests include A1C ≥6.5% (≥48 mmol/mol), fasting plasma glucose (FPG), 2-h plasma glucose (2-h PG) during a 75-g oral glucose tolerance test (OGTT), or random glucose accompanied by classic symptoms of hyperglycemia or hyperglycemic crises. 

Source: dc26s002.pdf


QUESTION 2
Who should be screened for Type 2 Diabetes?

EXPECTED SOURCE:
dc26s002.pdf

BASELINE RETRIEVED SOURCES:
dc26s002.pdf

BASELINE ANSWE

## Citation Accuracy

Citation accuracy was measured by checking whether each generated answer cited the expected ADA guideline source and avoided citing documents that were not provided in the retrieved context.

In [58]:
# EXTRACT PDF SOURCE NAMES FROM GENERATED ANSWERS


def extract_pdf_citations(answer):
    """
    Extract ADA PDF source names such as dc26s009.pdf
    from a generated answer.
    """

    if not isinstance(answer, str):
        return []

    citations = re.findall(
        r"dc26s\d{3}\.pdf",
        answer,
        flags=re.IGNORECASE
    )

    return sorted(
        set(citation.lower() for citation in citations)
    )

In [59]:
# EVALUATE CITATION ACCURACY


citation_results = []

for _, row in answer_evaluation_df.iterrows():

    expected_source = row["expected_source"].lower()


    # Baseline citations


    baseline_citations = extract_pdf_citations(
        row["baseline_answer"]
    )

    baseline_retrieved = [
        source.strip().lower()
        for source in row[
            "baseline_retrieved_sources"
        ].split(",")
    ]

    baseline_expected_cited = int(
        expected_source in baseline_citations
    )

    baseline_hallucinated_citation = int(
        any(
            citation not in baseline_retrieved
            for citation in baseline_citations
        )
    )

    baseline_citation_accurate = int(
        baseline_expected_cited == 1
        and baseline_hallucinated_citation == 0
    )


    # Re-ranked citations


    reranked_citations = extract_pdf_citations(
        row["reranked_answer"]
    )

    reranked_retrieved = [
        source.strip().lower()
        for source in row[
            "reranked_retrieved_sources"
        ].split(",")
    ]

    reranked_expected_cited = int(
        expected_source in reranked_citations
    )

    reranked_hallucinated_citation = int(
        any(
            citation not in reranked_retrieved
            for citation in reranked_citations
        )
    )

    reranked_citation_accurate = int(
        reranked_expected_cited == 1
        and reranked_hallucinated_citation == 0
    )

    citation_results.append({
        "category": row["category"],
        "question": row["question"],
        "expected_source": row["expected_source"],

        "baseline_citations": ", ".join(
            baseline_citations
        ),
        "baseline_expected_source_cited":
            baseline_expected_cited,
        "baseline_hallucinated_citation":
            baseline_hallucinated_citation,
        "baseline_citation_accurate":
            baseline_citation_accurate,

        "reranked_citations": ", ".join(
            reranked_citations
        ),
        "reranked_expected_source_cited":
            reranked_expected_cited,
        "reranked_hallucinated_citation":
            reranked_hallucinated_citation,
        "reranked_citation_accurate":
            reranked_citation_accurate
    })


citation_results_df = pd.DataFrame(
    citation_results
)

citation_results_df

,category,question,expected_source,baseline_citations,baseline_expected_source_cited,baseline_hallucinated_citation,baseline_citation_accurate,reranked_citations,reranked_expected_source_cited,reranked_hallucinated_citation,reranked_citation_accurate
0,Diagnosis,How is diabetes diagnosed?,dc26s002.pdf,dc26s002.pdf,1,0,1,dc26s002.pdf,1,0,1
1,Diagnosis,Who should be screened for Type 2 Diabetes?,dc26s002.pdf,dc26s002.pdf,1,0,1,dc26s002.pdf,1,0,1
2,Diagnosis,What tests are used to diagnose diabetes?,dc26s002.pdf,dc26s002.pdf,1,0,1,dc26s002.pdf,1,0,1
3,Glycemic targets,What is the recommended HbA1c target for most ...,dc26s006.pdf,dc26s006.pdf,1,0,1,dc26s006.pdf,1,0,1
4,Glycemic targets,How often should glycemic status be assessed?,dc26s006.pdf,dc26s006.pdf,1,0,1,dc26s006.pdf,1,0,1
5,Glycemic targets,When should glycemic goals be individualized?,dc26s006.pdf,dc26s006.pdf,1,0,1,dc26s006.pdf,1,0,1
6,Lifestyle,What lifestyle changes are recommended for Typ...,dc26s005.pdf,"dc26s005.pdf, dc26s009.pdf",1,0,1,dc26s005.pdf,1,0,1
7,Lifestyle,What physical activity is recommended for adul...,dc26s005.pdf,dc26s005.pdf,1,0,1,dc26s005.pdf,1,0,1
8,Lifestyle,What is the role of weight management in Type ...,dc26s005.pdf,dc26s005.pdf,1,0,1,"dc26s005.pdf, dc26s009.pdf",1,0,1
9,Medication,How does the ADA recommend selecting glucose-l...,dc26s009.pdf,dc26s009.pdf,1,0,1,dc26s009.pdf,1,0,1


In [60]:
# SUMMARIZE CITATION ACCURACY


citation_summary = pd.DataFrame({
    "System": [
        "Baseline retrieval + Qwen3-14B",
        "Re-ranked retrieval + Qwen3-14B"
    ],
    "Citation Accuracy": [
        citation_results_df[
            "baseline_citation_accurate"
        ].mean(),

        citation_results_df[
            "reranked_citation_accurate"
        ].mean()
    ],
    "Expected Source Citation Rate": [
        citation_results_df[
            "baseline_expected_source_cited"
        ].mean(),

        citation_results_df[
            "reranked_expected_source_cited"
        ].mean()
    ],
    "Hallucinated Citation Rate": [
        citation_results_df[
            "baseline_hallucinated_citation"
        ].mean(),

        citation_results_df[
            "reranked_hallucinated_citation"
        ].mean()
    ]
})

citation_summary[
    [
        "Citation Accuracy",
        "Expected Source Citation Rate",
        "Hallucinated Citation Rate"
    ]
] = citation_summary[
    [
        "Citation Accuracy",
        "Expected Source Citation Rate",
        "Hallucinated Citation Rate"
    ]
].round(3)

citation_summary

,System,Citation Accuracy,Expected Source Citation Rate,Hallucinated Citation Rate
0,Baseline retrieval + Qwen3-14B,1.0,1.0,0.0
1,Re-ranked retrieval + Qwen3-14B,1.0,1.0,0.0


### Observation

Both systems achieved a citation accuracy of 1.00 across the 15 evaluation questions. Every answer cited the expected ADA guideline document, and neither system generated a citation to a document that was not included in the retrieved context. This shows that both versions of the system produced consistent source citations.

However, this metric only checks whether the correct document name was cited. It does not confirm that every individual statement in the answer was fully supported by the cited excerpt.

## Human Answer-Quality Evaluation

Each answer was manually reviewed and rated using the following criteria.

- **Relevance:** How directly the answer addresses the question.
- **Faithfulness:** How well the answer is supported by the retrieved ADA excerpts.
- **Clarity:** How clear, concise, and understandable the response is.

Each criterion was rated from 1 to 5:

- 1 = Poor
- 2 = Below average
- 3 = Acceptable
- 4 = Good
- 5 = Excellent

In [61]:
# CREATE HUMAN-RATING SHEET


human_rating_df = answer_evaluation_df[
    [
        "category",
        "question",
        "expected_source",
        "baseline_answer",
        "reranked_answer"
    ]
].copy()

human_rating_df[
    "baseline_relevance"
] = None

human_rating_df[
    "baseline_faithfulness"
] = None

human_rating_df[
    "baseline_clarity"
] = None

human_rating_df[
    "reranked_relevance"
] = None

human_rating_df[
    "reranked_faithfulness"
] = None

human_rating_df[
    "reranked_clarity"
] = None

human_rating_df

,category,question,expected_source,baseline_answer,reranked_answer,baseline_relevance,baseline_faithfulness,baseline_clarity,reranked_relevance,reranked_faithfulness,reranked_clarity
0,Diagnosis,How is diabetes diagnosed?,dc26s002.pdf,Diabetes is diagnosed by demonstrating increas...,Diabetes is diagnosed by demonstrating increas...,None,None,None,None,None,None
1,Diagnosis,Who should be screened for Type 2 Diabetes?,dc26s002.pdf,Children and adolescents with overweight (BMI ...,Asymptomatic adults with overweight or obesity...,None,None,None,None,None,None
2,Diagnosis,What tests are used to diagnose diabetes?,dc26s002.pdf,Diabetes can be diagnosed using A1C or plasma ...,Diabetes can be diagnosed using A1C or plasma ...,None,None,None,None,None,None
3,Glycemic targets,What is the recommended HbA1c target for most ...,dc26s006.pdf,The recommended HbA1c target for most adults w...,The recommended HbA1c target for most adults w...,None,None,None,None,None,None
4,Glycemic targets,How often should glycemic status be assessed?,dc26s006.pdf,Glycemic status should be assessed at least tw...,Glycemic status should be assessed at least tw...,None,None,None,None,None,None
5,Glycemic targets,When should glycemic goals be individualized?,dc26s006.pdf,Glycemic goals should be individualized based ...,Glycemic goals should be individualized based ...,None,None,None,None,None,None
6,Lifestyle,What lifestyle changes are recommended for Typ...,dc26s005.pdf,Lifestyle changes recommended for Type 2 Diabe...,Lifestyle intervention programs should be inte...,None,None,None,None,None,None
7,Lifestyle,What physical activity is recommended for adul...,dc26s005.pdf,Adults with diabetes should be counseled to re...,Adults with diabetes should be counseled to re...,None,None,None,None,None,None
8,Lifestyle,What is the role of weight management in Type ...,dc26s005.pdf,Weight management plays a significant role in ...,Weight management is a distinct treatment goal...,None,None,None,None,None,None
9,Medication,How does the ADA recommend selecting glucose-l...,dc26s009.pdf,The ADA recommends using a person-centered sha...,The ADA recommends selecting glucose-lowering ...,None,None,None,None,None,None


In [62]:
# ADD MANUAL HUMAN RATINGS


# Ratings use a 1–5 scale:
# 1 = Poor
# 2 = Below average
# 3 = Acceptable
# 4 = Good
# 5 = Excellent

manual_ratings = [
    # Q1: How is diabetes diagnosed?
    [4, 5, 5, 5, 5, 5],

    # Q2: Who should be screened?
    [3, 5, 4, 5, 5, 5],

    # Q3: Diagnostic tests
    [5, 5, 5, 5, 5, 5],

    # Q4: HbA1c target
    [4, 5, 5, 5, 5, 5],

    # Q5: Frequency of glycemic assessment
    [5, 5, 5, 5, 5, 5],

    # Q6: Individualized glycemic goals
    [4, 5, 4, 5, 5, 5],

    # Q7: Lifestyle changes
    [3, 4, 4, 4, 4, 4],

    # Q8: Physical activity
    [3, 5, 5, 4, 5, 5],

    # Q9: Weight management
    [5, 5, 4, 5, 5, 4],

    # Q10: Medication selection
    [5, 5, 5, 5, 5, 5],

    # Q11: Combination therapy
    [4, 5, 5, 5, 5, 5],

    # Q12: Medication-choice factors
    [4, 5, 4, 5, 5, 5],

    # Q13: Cardiovascular risk
    [4, 5, 4, 5, 5, 5],

    # Q14: Blood pressure management
    [5, 5, 4, 5, 5, 4],

    # Q15: Lipid management
    [4, 5, 4, 5, 5, 4]
]

rating_columns = [
    "baseline_relevance",
    "baseline_faithfulness",
    "baseline_clarity",
    "reranked_relevance",
    "reranked_faithfulness",
    "reranked_clarity"
]

for column_index, column_name in enumerate(rating_columns):
    human_rating_df[column_name] = [
        row[column_index]
        for row in manual_ratings
    ]

human_rating_df[
    [
        "category",
        "question",
        "baseline_relevance",
        "baseline_faithfulness",
        "baseline_clarity",
        "reranked_relevance",
        "reranked_faithfulness",
        "reranked_clarity"
    ]
]

,category,question,baseline_relevance,baseline_faithfulness,baseline_clarity,reranked_relevance,reranked_faithfulness,reranked_clarity
0,Diagnosis,How is diabetes diagnosed?,4,5,5,5,5,5
1,Diagnosis,Who should be screened for Type 2 Diabetes?,3,5,4,5,5,5
2,Diagnosis,What tests are used to diagnose diabetes?,5,5,5,5,5,5
3,Glycemic targets,What is the recommended HbA1c target for most ...,4,5,5,5,5,5
4,Glycemic targets,How often should glycemic status be assessed?,5,5,5,5,5,5
5,Glycemic targets,When should glycemic goals be individualized?,4,5,4,5,5,5
6,Lifestyle,What lifestyle changes are recommended for Typ...,3,4,4,4,4,4
7,Lifestyle,What physical activity is recommended for adul...,3,5,5,4,5,5
8,Lifestyle,What is the role of weight management in Type ...,5,5,4,5,5,4
9,Medication,How does the ADA recommend selecting glucose-l...,5,5,5,5,5,5


In [63]:
# SUMMARIZE HUMAN ANSWER-QUALITY RATINGS


human_rating_summary = pd.DataFrame({
    "System": [
        "Baseline retrieval + Qwen3-14B",
        "Re-ranked retrieval + Qwen3-14B"
    ],

    "Average Relevance": [
        human_rating_df["baseline_relevance"].mean(),
        human_rating_df["reranked_relevance"].mean()
    ],

    "Average Faithfulness": [
        human_rating_df["baseline_faithfulness"].mean(),
        human_rating_df["reranked_faithfulness"].mean()
    ],

    "Average Clarity": [
        human_rating_df["baseline_clarity"].mean(),
        human_rating_df["reranked_clarity"].mean()
    ]
})

human_rating_summary[
    [
        "Average Relevance",
        "Average Faithfulness",
        "Average Clarity"
    ]
] = human_rating_summary[
    [
        "Average Relevance",
        "Average Faithfulness",
        "Average Clarity"
    ]
].round(2)

human_rating_summary

,System,Average Relevance,Average Faithfulness,Average Clarity
0,Baseline retrieval + Qwen3-14B,4.13,4.93,4.47
1,Re-ranked retrieval + Qwen3-14B,4.87,4.93,4.73


### Observation

The re-ranked system received higher average ratings for relevance and clarity than the baseline system. Faithfulness ratings remained high and nearly identical for both systems because, during manual review, the generated responses were generally consistent with the retrieved ADA guideline excerpts.

The greatest improvements were observed for questions related to screening, medication selection, combination therapy, and lipid management, where the re-ranked system typically produced more complete and informative responses. However, re-ranking did not improve every answer. Questions requiring information from multiple guideline sections, particularly those related to lifestyle management, remained more challenging, with some responses covering only part of the expected recommendations.

## Qualitative Analysis

The evaluation identified several patterns in the system's performance.

### Improved completeness

The re-ranked system generally produced more complete answers for several questions. For example, the screening response included recommendations for adults, children, and adolescents, whereas the baseline response focused primarily on children and adolescents. The re-ranked system also provided more complete responses for medication selection, combination therapy, and lipid management.

### Similar performance

For questions with highly specific wording, such as the HbA1c target and glycemic-assessment frequency, both systems produced similar, guideline-consistent responses. In these cases, Cross-Encoder re-ranking provided little additional benefit.

### Remaining limitations

Questions requiring information from multiple guideline sections remained more challenging for both systems. Some lifestyle-related responses addressed only part of the expected recommendations, suggesting that broader questions may benefit from improved document chunking or retrieval from multiple guideline sections.

### Overall finding

Cross-Encoder re-ranking improved retrieval precision and generally produced more relevant and complete answers. However, the improvement was modest and was not observed for every question. Overall, the results suggest that re-ranking is a useful enhancement, although retrieval quality remains influenced by document chunking and the breadth of the user's question.